# KPIs y Métricas de Negocio en Data Science

En este cuaderno exploraremos cómo calcular y analizar métricas clave para la toma de decisiones empresariales utilizando Python. Cubriremos tres áreas fundamentales:
1. **Análisis de Cohortes (Cohort Analysis)**: Retención de usuarios a lo largo del tiempo.
2. **Embudos de Conversión (Funnels)**: Identificación de puntos de fricción en los procesos de conversión.
3. **Segmentación de Clientes (RFM)**: Recencia, Frecuencia y Monetización.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo
sns.set_theme(style='whitegrid')
print('Bibliotecas importadas correctamente.')

--- 
## 1. Análisis de Cohortes (Cohort Analysis)

El análisis de cohortes permite entender el comportamiento de grupos de usuarios que comparten una característica común (por ejemplo, el mes de su primera compra) a lo largo del tiempo.

In [ ]:
# Generación de datos sintéticos de transacciones
pd.np.random.seed(42)
n_transactions = 10000

user_ids = np.random.randint(1000, 2000, size=n_transactions)
dates = pd.date_range(start='2023-01-01', end='2023-12-31')
transaction_dates = np.random.choice(dates, size=n_transactions)
amounts = np.random.exponential(scale=50, size=n_transactions) + 10

df = pd.DataFrame({
    'CustomerID': user_ids,
    'InvoiceDate': transaction_dates,
    'Amount': amounts
})

# Filtrar para obtener el mes de la primera compra (CohortMonth)
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')
df['CohortMonth'] = df.groupby('CustomerID')['InvoiceMonth'].transform('min')

print(df.head())

In [ ]:
# Calcular el índice de cohorte (número de meses desde la primera compra)
def get_month_int(d1, d2):
    return (d1.year - d2.year) * 12 + (d1.month - d2.month)

df['CohortIndex'] = df.apply(lambda row: get_month_int(row['InvoiceMonth'], row['CohortMonth']), axis=1)

# Agrupar para obtener el recuento de usuarios únicos por cohorte e índice
cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].apply(lambda x: len(x.unique())).reset_index()
cohort_counts = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')

# Calcular porcentajes de retención
cohort_sizes = cohort_counts.iloc[:, 0]
retention = cohort_counts.divide(cohort_sizes, axis=0)

print(retention.round(3)*100)

In [ ]:
# Visualización de la matriz de retención
plt.figure(figsize=(12, 8))
plt.title('Matriz de Retención por Cohorte (%)')
sns.heatmap(retention * 100, annot=True, fmt='.1f', cmap='Blues')
plt.xlabel('Índice de Mes')
plt.ylabel('Mes de Cohorte')
plt.show()

--- 
## 2. Embudos de Conversión (Funnels)

Un embudo de conversión ayuda a rastrear los pasos que siguen los usuarios para completar un objetivo y detectar dónde se abandonan.

In [ ]:
# Datos sintéticos para el embudo
stages = ['Visitas', 'Registro', 'Añadir al Carrito', 'Compra']
users_count = [50000, 15000, 5000, 1200]

funnel_df = pd.DataFrame({
    'Stage': stages,
    'Users': users_count
})

# Calcular tasa de conversión respecto al paso anterior y al inicio
funnel_df['Conversion_Total'] = (funnel_df['Users'] / funnel_df['Users'].iloc[0]) * 100
funnel_df['Conversion_Step'] = (funnel_df['Users'] / funnel_df['Users'].shift(1)) * 100
funnel_df['Conversion_Step'].fillna(100.0, inplace=True)

print(funnel_df)

In [ ]:
# Gráfico simple del embudo
plt.figure(figsize=(10, 5))
sns.barplot(x='Users', y='Stage', data=funnel_df, palette='viridis')
plt.title('Embudo de Conversión de Clientes')
plt.xlabel('Número de Usuarios')
plt.ylabel('Etapa')
plt.show()

--- 
## 3. Segmentación de Clientes (Modelo RFM)

El análisis RFM (**Recency, Frequency, Monetary**) clasifica a los clientes según:
- **Recencia (R)**: ¿Cuándo fue su última compra?
- **Frecuencia (F)**: ¿Con qué frecuencia compran?
- **Monetario (M)**: ¿Cuánto dinero gastan?

In [ ]:
# Calcular métricas RFM
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,
    'InvoiceMonth': 'count',
    'Amount': 'sum'
}).rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceMonth': 'Frequency',
    'Amount': 'Monetary'
})

# Asignar puntuaciones de 1 a 4 usando cuartiles
r_labels = range(4, 0, -1)
f_labels = range(1, 5)
m_labels = range(1, 5)

rfm['R'] = pd.qcut(rfm['Recency'], q=4, labels=r_labels)
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=f_labels)
rfm['M'] = pd.qcut(rfm['Monetary'], q=4, labels=m_labels)

rfm['RFM_Score'] = rfm[['R', 'F', 'M']].sum(axis=1)

print(rfm.head())

### Conclusiones

Con este cuaderno hemos calculado métricas esenciales para medir la salud del negocio: la retención mediante cohortes, la eficiencia operativa con embudos y la priorización de clientes mediante RFM.